# Desafio: Modelando um Agente Preditor de Consumo

**Gabarito de referência — sequência completa do roteiro, com visualizações**

*Aula 04 · Inteligência Artificial e Aprendizagem de Máquina · FECAP · 2026/02*

> Este notebook é uma resolução completa de exemplo, seguindo os 9 checkpoints do desafio. Serve como referência para conduzir a aula — não é o gabarito único (cada dupla pode fazer escolhas diferentes e válidas).

## Setup — importações e carregamento

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Checkpoint 1 — Explorar os dados

Carregamos o CSV e olhamos: quantas linhas, quais colunas, tem nulo, e a correlação de cada variável numérica com `mpg`.

In [ ]:
df = pd.read_csv("mpg_desafio_alunos.csv")

print(df.shape)
df.info()

In [ ]:
df.corr(numeric_only=True)["mpg"].sort_values()

**Visualizando a relação mais forte (`weight`) antes de decidir:**

In [ ]:
sns.scatterplot(x="weight", y="mpg", data=df)
plt.title("Peso x Consumo")
plt.xlabel("Peso (weight)")
plt.ylabel("Consumo (mpg)")
plt.show()

**Leitura:** `weight` tem a correlação mais forte com `mpg` (negativa — carro mais pesado, menos econômico). `horsepower` e `cylinders` também são fortes, mas um pouco menores.

## Checkpoint 2 — Escolher a feature

Antes de decidir, comparamos as 3 candidatas lado a lado — não só pelo número de correlação, mas pela forma da relação.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, coluna in zip(axes, ["horsepower", "weight", "cylinders"]):
    sns.scatterplot(x=coluna, y="mpg", data=df, ax=ax)
    ax.set_title(f"{coluna} x mpg")
plt.tight_layout()
plt.show()

Com base na correlação **e** na forma visual da relação, escolhemos **`weight`**.

In [ ]:
X = df[["weight"]]
y = df["mpg"]

## Checkpoint 3 — Tratar valores nulos

`weight` não tem nulos — mas `horsepower` tem 6. Como não vamos usar `horsepower` como feature aqui, não precisamos tratar nada neste exemplo. (Se a dupla escolher `horsepower`, precisa limpar antes de continuar.)

In [ ]:
df["weight"].isnull().sum()

## Checkpoint 4 — Separar treino e teste

Cada dupla escolhe o `random_state` que quiser. Aqui usamos `7`, só como exemplo — o valor em si não importa para o aprendizado.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7
)

print(f"Treino: {len(X_train)} carros")
print(f"Teste: {len(X_test)} carros")

## Checkpoint 5 — Treinar a regressão linear

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)

print("w1:", modelo.coef_[0])
print("w0:", modelo.intercept_)

**Visualizando a reta que o modelo encontrou:**

In [ ]:
x_linha = pd.DataFrame({"weight": np.linspace(df["weight"].min(), df["weight"].max(), 100)})
y_linha_prevista = modelo.predict(x_linha)

sns.scatterplot(x="weight", y="mpg", data=df)
plt.plot(x_linha, y_linha_prevista, color="green", label="Reta do modelo")
plt.legend()
plt.title("Reta ajustada — weight x mpg")
plt.show()

## Checkpoint 6 — Avaliar (MAE, MSE, R²)

Avaliação no **conjunto de teste da própria dupla** — só para iterar, ainda não é o resultado oficial.

In [ ]:
pred = modelo.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("MSE:", mean_squared_error(y_test, pred))
print("R²:", r2_score(y_test, pred))

**Visualizando o erro — previsto vs. real:**

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, pred, alpha=0.7)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lims, lims, color="red", linestyle="--", label="Previsão perfeita")
plt.xlabel("Valor real")
plt.ylabel("Valor previsto")
plt.title("Previsto vs Real (teste da dupla)")
plt.legend()
plt.show()

**Como ler:** quanto mais perto os pontos estiverem da linha vermelha (previsão perfeita), melhor o modelo. Pontos muito espalhados indicam erro grande.

## Checkpoint 7 — Tentar melhorar (polinomial)

Testamos grau 2 e grau 3, comparando o R² de cada um contra a reta simples.

In [ ]:
for grau in [1, 2, 3]:
    poly = PolynomialFeatures(degree=grau)
    Xtr_p = poly.fit_transform(X_train)
    Xte_p = poly.transform(X_test)
    modelo_p = LinearRegression().fit(Xtr_p, y_train)
    r2_p = r2_score(y_test, modelo_p.predict(Xte_p))
    print(f"Grau {grau}: R² = {r2_p:.4f}")

**Visualizando reta vs. curva, lado a lado nos mesmos pontos:**

In [ ]:
poly2 = PolynomialFeatures(degree=2)
Xtr_p2 = poly2.fit_transform(X_train)
modelo_p2 = LinearRegression().fit(Xtr_p2, y_train)

poly3 = PolynomialFeatures(degree=3)
Xtr_p3 = poly3.fit_transform(X_train)
modelo_p3 = LinearRegression().fit(Xtr_p3, y_train)

x_linha_poly = poly2.transform(x_linha)
y_pred_poly = modelo_p2.predict(x_linha_poly)

sns.scatterplot(x="weight", y="mpg", data=df)
plt.plot(x_linha, y_linha_prevista, color="gray", label="Reta (grau 1)")
plt.plot(x_linha, y_pred_poly, color="green", label="Curva (grau 2)")
plt.plot(x_linha, y_pred_poly, color="blue", label="Curva (grau 3)")
plt.legend()
plt.title("Reta vs. curva polinomial")
plt.show()

## Checkpoint 8 — Decidir e justificar

**Decisão deste exemplo:** ficamos com a regressão **linear simples (grau 1)**.

**Justificativa:** como o gráfico do Checkpoint 7 mostra, a curva de grau 2 quase não se distingue visualmente da reta — o ganho de R² foi pequeno. Usar o modelo mais simples, quando o ganho do mais complexo é marginal, reduz o risco de overfitting sem perder poder preditivo. Também comparamos R² do treino vs. teste para confirmar que não há sinal de overfitting.

In [ ]:
r2_treino = r2_score(y_train, modelo.predict(X_train))
r2_teste = r2_score(y_test, modelo.predict(X_test))

print(f"R² no treino: {r2_treino:.4f}")
print(f"R² no teste: {r2_teste:.4f}")

## Checkpoint 9 — Avaliação final

Só agora carregamos o conjunto **nunca visto** (`mpg_desafio_final.csv`) e aplicamos o modelo já treinado e decidido.

In [ ]:
df_final = pd.read_csv("mpg_desafio_final.csv")
X_final = df_final[["weight"]]
y_final = df_final["mpg"]

pred_final = modelo.predict(X_final)
r2_final = r2_score(y_final, pred_final)

print(f"R² OFICIAL: {r2_final:.4f}")

**Visualizando o resultado oficial — previsto vs. real:**

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_final, pred_final, alpha=0.7, color="green")
lims = [min(y_final.min(), pred_final.min()), max(y_final.max(), pred_final.max())]
plt.plot(lims, lims, color="red", linestyle="--", label="Previsão perfeita")
plt.xlabel("Valor real")
plt.ylabel("Valor previsto")
plt.title("Previsto vs Real (conjunto OFICIAL)")
plt.legend()
plt.show()

---
## Debrief — por que o R² de cada dupla pode variar tanto?

Demonstração: rodamos o **mesmo modelo** (mesma feature, mesmo grau) em 30 `random_state` diferentes, e comparamos o R² "próprio" (medido no teste de cada rodada) contra o R² "oficial" (sempre no mesmo `mpg_desafio_final.csv`).

In [ ]:
r2_proprios = []
r2_oficiais = []

for seed in range(30):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    m = LinearRegression().fit(Xtr, ytr)
    r2_proprios.append(r2_score(yte, m.predict(Xte)))
    r2_oficiais.append(r2_score(y_final, m.predict(X_final)))

r2_proprios = np.array(r2_proprios)
r2_oficiais = np.array(r2_oficiais)

print(f"R² PRÓPRIO   -> média: {r2_proprios.mean():.3f}   desvio: {r2_proprios.std():.3f}")
print(f"R² OFICIAL   -> média: {r2_oficiais.mean():.3f}   desvio: {r2_oficiais.std():.3f}")

In [ ]:
plt.figure(figsize=(9,5))
plt.hist(r2_proprios, bins=10, alpha=0.6, label="R² próprio (disperso)", color="orange")
plt.hist(r2_oficiais, bins=10, alpha=0.6, label="R² oficial (estável)", color="green")
plt.xlabel("R²")
plt.ylabel("Frequência (de 30 seeds)")
plt.title("Dispersão do R²: próprio vs. oficial")
plt.legend()
plt.show()

**Conclusão para o debrief com a turma:** o R² medido no conjunto de teste de cada dupla mistura duas fontes de sorte (qual modelo foi treinado + qual "prova" caiu no teste). O R² oficial, medido sempre no mesmo conjunto fixo, isola só a primeira — por isso é mais estável e mais confiável como medida real de qualidade do modelo.